# Project Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sqlalchemy

In [2]:
print(sqlalchemy.__version__)

2.0.48


In [3]:
from sqlalchemy import create_engine, inspect
from sqlalchemy import text
from sqlalchemy.schema import CreateSchema
from sqlalchemy import BigInteger, Column, Integer, String, Float, Boolean
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column

# Create SQL Engine and Test Connection

In [4]:
engine = create_engine('postgresql+psycopg2://postgres@localhost/temp_country_club')

In [5]:
with engine.connect() as conn:
    result = conn.execute(text("select count(*) from public.bookings"))
    print(result.fetchone())

(4043,)


In [6]:
bookings = pd.read_sql_table('bookings', con=engine)

In [7]:
facilities = pd.read_sql_table('facilities', con=engine)

In [8]:
members = pd.read_sql_table('members', con=engine) 

In [9]:
bookings.head()

,bookid,facid,memid,starttime,slots
0,0,3,1,2012-07-03 11:00:00,2
1,1,4,1,2012-07-03 08:00:00,2
2,2,6,0,2012-07-03 18:00:00,2
3,3,7,1,2012-07-03 19:00:00,2
4,4,8,1,2012-07-03 10:00:00,1


In [10]:
query = """
with recommenders as (
        select
                memid,
                firstname,
                surname,
                nullif(recommendedby, '')::int as recid
        from members)
select
        concat(r.surname, ', ', r.firstname) as membername,
                case
                        when r.recid is not null then concat(m.surname, ', ', m.firstname)
                        when r.recid  is null then null
                end as recommender
from recommenders as r
left join members as m on r.recid = m.memid
order by membername asc;
"""

In [11]:
members_recommenders = pd.read_sql_query(query, engine)

In [12]:
members_recommenders.head()

,membername,recommender
0,"Bader, Florence","Stibbons, Ponder"
1,"Baker, Anne","Stibbons, Ponder"
2,"Baker, Timothy","Farrell, Jemima"
3,"Boothe, Tim","Rownam, Tim"
4,"Butters, Gerald","Smith, Darren"


In [14]:
query = """
with book_lim as (
        select
                b.bookid,
                b.facid,
                b.starttime,
                b.memid
        from bookings as b
        where b.memid != 0
),
usage_list as (
        select
                concat(f.name, ': ', m.firstname, ', ', m.surname) as list
        from facilities as f
        left join book_lim as b on b.facid = f.facid
        left join members as m on b.memid = m.memid
        group by list, b.bookid
        order by list
)
select
        u.list,
        count(*) as total_each
from usage_list as u
group by u.list
order by u.list;
"""

In [15]:
usage_by_member = pd.read_sql_query(query, engine)

In [16]:
usage_by_member.head()

,list,total_each
0,"Badminton Court: Anna, Mackenzie",30
1,"Badminton Court: Anne, Baker",10
2,"Badminton Court: Burton, Tracy",2
3,"Badminton Court: Charles, Owen",6
4,"Badminton Court: Darren, Smith",132


In [17]:
query = """
with book_lim as (
	select
		b.bookid,
		b.facid,
		to_char(b.starttime::timestamp, 'YYYY-MM') as y_m
		/*b.memid*/
	from bookings as b
	where b.memid != 0
),
usage_list as (
	select	
		concat(f.name, ': ', b.y_m) as list
	from facilities as f
	left join book_lim as b on b.facid = f.facid
/*	left join members as m on b.memid = m.memid*/
	group by list, b.bookid
	order by list
)
select
	u.list,
	count(*) as total_each
from usage_list as u
group by u.list
order by u.list;
"""

In [20]:
usage_by_month = pd.read_sql_query(query, engine)

In [21]:
usage_by_month.head()

,list,total_each
0,Badminton Court: 2012-07,51
1,Badminton Court: 2012-08,132
2,Badminton Court: 2012-09,161
3,Massage Room 1: 2012-07,77
4,Massage Room 1: 2012-08,153
